# State-level household health insurance coverage

This extension adds a state/UT indicator of household reach of health insurance and financing schemes from NFHS-5 (2019–21). It measures whether at least one usual household member is covered. It is a financial-protection proxy, not a direct measure of affordability, benefit adequacy, claim payment or out-of-pocket expenditure.

The correlation with per-capita NSDP is descriptive and cross-sectional. The income series is FY2021–22 at current prices; three states/UTs without a separate NSDP value are excluded from the correlation rather than imputed.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

DATA = Path('../Dataset')
OUT = Path('../Visualisations')
insurance = pd.read_csv(DATA / 'state_health_insurance_coverage_nfhs5.csv')
socio = pd.read_csv(DATA / 'state_socioeconomic_covariates_nfhs5_rbi.csv')

assert insurance.state_ut.is_unique and len(insurance) == 36
assert insurance.households_any_member_covered_pct.between(0, 100).all()
matched = insurance.merge(socio[['state_ut', 'per_capita_nsdp_2021_22_inr']], on='state_ut', how='left', validate='one_to_one')
complete = matched.dropna(subset=['per_capita_nsdp_2021_22_inr']).copy()
r, p = pearsonr(complete.per_capita_nsdp_2021_22_inr, complete.households_any_member_covered_pct)
rho, p_rho = spearmanr(complete.per_capita_nsdp_2021_22_inr, complete.households_any_member_covered_pct)
print(f'Coverage observations: {len(insurance)} state/UTs')
print(f'Coverage–NSDP complete cases: {len(complete)}')
print(f'Pearson r (coverage, nominal per-capita NSDP): {r:.3f}; p={p:.3f}')
print(f'Spearman rho: {rho:.3f}; p={p_rho:.3f}')
print(f'Coverage range: {insurance.households_any_member_covered_pct.min():.1f}%–{insurance.households_any_member_covered_pct.max():.1f}%')


## Interpretation

NFHS-5 reports an all-India estimate of 41.0% of households with at least one usual member covered. Across states/UTs, published coverage ranges from 1.8% in Andaman & Nicobar Islands to 87.8% in Rajasthan. The state-level correlation with nominal per-capita NSDP is close to zero in the available 33-state/UT complete-case comparison (Pearson r approximately 0.052; p approximately 0.776). This does not establish that income is unrelated to financial protection: the comparison is small, cross-sectional, unweighted and sensitive to the limitations of the two indicators.

Insurance coverage is not equivalent to effective financial protection. It does not show whether a household member needed care, used a covered provider, received a claim payment, faced exclusions or still incurred substantial out-of-pocket costs.

In [ ]:
plot_data = insurance.sort_values('households_any_member_covered_pct', ascending=True)
fig, ax = plt.subplots(figsize=(10, 11))
ax.barh(plot_data.state_ut, plot_data.households_any_member_covered_pct)
ax.set_xlim(0, 100)
ax.set_xlabel('Households with at least one usual member covered (%)')
ax.set_title('NFHS-5 household health insurance / financing coverage, 2019–21')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / '12_nfhs5_household_health_insurance_coverage.svg', format='svg', bbox_inches='tight')
plt.show()


## Limitations and source

The indicator includes any health insurance or financing scheme and does not distinguish public from private coverage in the headline percentage. It is a household-level reach measure; it does not measure household spending or the adequacy of the cover. The per-capita NSDP series is nominal and from FY2021–22, while NFHS-5 fieldwork was in 2019–21. Three observations are omitted from the correlation because the NSDP series has no separate value. Source: IIPS and ICF, *NFHS-5 India Report*, Table 11.15, p. 476: https://dhsprogram.com/pubs/pdf/FR375/FR375.pdf.